# Iris: классификация трёх видов цветка

В этом примере сеть классифицирует цветки ириса по четырём измерениям: длине и ширине чашелистика и лепестка. Набор содержит три вида цветка. Мы разделим его на обучающую и тестовую части, обучим многослойную сеть и оценим её на отложенных примерах.

Покажем также подготовку признаков: масштабирование по статистикам только обучающей выборки и one-hot кодирование классов для categorical cross-entropy.


In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from open_neuron.activations import softmax, tanh
from open_neuron.layers import Layer
from open_neuron.models import NeuralNetwork
from open_neuron.optimizers import Adam

np.set_printoptions(precision=3, suppress=True)
iris = load_iris()
X = iris.data
y = iris.target
class_names = iris.target_names

print("Признаки:", iris.feature_names)
print("Классы:", class_names)
print("Размер данных:", X.shape)


Признаки: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Классы: ['setosa' 'versicolor' 'virginica']
Размер данных: (150, 4)


## Делим данные и масштабируем признаки

`stratify=y` сохраняет доли трёх видов в обеих выборках, а `random_state` делает разбиение повторяемым. Масштабировщик обучается только на `X_train`; к тестовым данным применяется уже найденное преобразование. Так сведения из тестовой выборки не участвуют в подготовке обучения.

Метки классов хранятся целыми числами. Для сети преобразуем только обучающие метки в one-hot матрицу: например, класс `2` станет `[0, 0, 1]`.


In [2]:
X_train, X_test, y_train_labels, y_test_labels = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train = np.eye(len(class_names))[y_train_labels]

print("Обучающих примеров:", len(X_train))
print("Тестовых примеров:  ", len(X_test))
print("Средние обучающих признаков после масштабирования:", X_train_scaled.mean(axis=0))
print("One-hot метки первых трёх примеров:")
print(y_train[:3])


Обучающих примеров: 120
Тестовых примеров:   30
Средние обучающих признаков после масштабирования: [-0. -0.  0.  0.]
One-hot метки первых трёх примеров:
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


## Строим и обучаем сеть

Скрытый слой содержит четыре нейрона с `tanh`. Выходной слой содержит по одному нейрону на каждый вид ириса; `softmax` превращает их значения в распределение вероятностей. Категориальная кросс-энтропия сравнивает это распределение с one-hot ответами.

Во время обучения сеть видит только масштабированные обучающие примеры. Тестовую выборку не передаём в `fit`; используем её один раз после обучения для оценки обобщения.


In [3]:
np.random.seed(7)
model = NeuralNetwork(
    [
        Layer(4, activation=tanh),
        Layer(len(class_names), activation=softmax),
    ],
    inputs_dim=X_train_scaled.shape[1],
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.01),
)

model.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=16,
    shuffle=True,
)


Training started with Overall Loss 1.3363 on Train Data
Epoch 100/100, Loss: 0.0474, Validation Loss: 0.0467 on Train Data
Training completed with Overall Loss 0.0467


## Оцениваем на отложенной выборке

`argmax` выбирает вид с наибольшей предсказанной вероятностью. Матрица ошибок показывает, какие виды сеть путает; отчёт отдельно показывает precision, recall и F1 для каждого вида. Результат относится к конкретному фиксированному разбиению, а не гарантирует такую же точность на любом наборе данных.


In [4]:
probabilities = model.predict(X_test_scaled)
predicted_labels = np.argmax(probabilities, axis=1)

print(f"Точность на тестовой выборке: {accuracy_score(y_test_labels, predicted_labels):.1%}")
print("Матрица ошибок (строки — ответ, столбцы — предсказание):")
print(confusion_matrix(y_test_labels, predicted_labels, labels=np.arange(len(class_names))))
print("Отчёт по классам:")
print(classification_report(
    y_test_labels,
    predicted_labels,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
))

print("Первые предсказания:")
for expected, predicted, probabilities_for_sample in zip(
    y_test_labels[:5], predicted_labels[:5], probabilities[:5]
):
    print(
        f"ответ: {class_names[expected]}, "
        f"предсказание: {class_names[predicted]}, "
        f"вероятности: {probabilities_for_sample}"
    )


Точность на тестовой выборке: 96.7%
Матрица ошибок (строки — ответ, столбцы — предсказание):
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
Отчёт по классам:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Первые предсказания:
ответ: setosa, предсказание: setosa, вероятности: [0.995 0.005 0.   ]
ответ: virginica, предсказание: virginica, вероятности: [0.001 0.203 0.796]
ответ: versicolor, предсказание: versicolor, вероятности: [0.01 0.99 0.  ]
ответ: versicolor, предсказание: versicolor, вероятности: [0.006 0.994 0.   ]
ответ: setosa, предсказание: setosa, вероятности: [0.995 0.005 0.   ]


In [5]:
model.summary()

Info:
Network type: "Classification", layers: 2, neurons: 7, optimizer: "Adam", epochs: 100, batch size: 16, loss function: "Categorical Crossentropy"
Last forward:
Layer 1, neurons: 4, inputs: [-0.766  0.787 -1.346 -1.323], activation: tanh, output (a): [ 0.996  0.998 -0.993 -1.   ]
Unit 1/4, inputs: [-0.766  0.787 -1.346 -1.323], weights: [-0.815  1.5   -0.587 -0.294], bias: [0.128], z: 3.1122440922529346, activation: tanh, a: 0.9960471537217452
Unit 2/4, inputs: [-0.766  0.787 -1.346 -1.323], weights: [ 0.235 -0.112 -0.81  -1.421], bias: [0.745], z: 3.4464806937474552, activation: tanh, a: 0.9979722502455952
Unit 3/4, inputs: [-0.766  0.787 -1.346 -1.323], weights: [ 0.846 -0.817  1.453  0.796], bias: [1.5], z: -2.8005535538822177, activation: tanh, a: -0.9926396433830819
Unit 4/4, inputs: [-0.766  0.787 -1.346 -1.323], weights: [ 0.063 -0.535  1.375  1.596], bias: [-2.159], z: -6.590849177173231, activation: tanh, a: -0.9999962304429567
Layer 2, neurons: 3, inputs: [ 0.996  0.998 -